In [1]:
import pandas as pd
import re

rsa_path = "/content/missense_mutations_with_RSA.csv"
alphamissense_path = "/content/AF-P18510-F1-aa-substitutions.csv"

rsa = pd.read_csv(rsa_path)
am = pd.read_csv(alphamissense_path)

print("RSA columns:", list(rsa.columns))
print("AlphaMissense columns:", list(am.columns))
print("RSA rows:", len(rsa), "AlphaMissense rows:", len(am))
rsa.head()

RSA columns: ['gnomAD ID', 'Position', 'HGVS Consequence', 'Protein Consequence', 'ResNum', 'WT_AA', 'AA', 'RSA', 'ACC', 'Allele Frequency', 'Allele Count', 'ClinVar Germline Classification']
AlphaMissense columns: ['protein_variant', 'am_pathogenicity', 'am_class']
RSA rows: 266 AlphaMissense rows: 3363


,gnomAD ID,Position,HGVS Consequence,Protein Consequence,ResNum,WT_AA,AA,RSA,ACC,Allele Frequency,Allele Count,ClinVar Germline Classification
0,2-113118022-G-T,113118022,p.Ala2Ser,p.Ala2Ser,2,Ala,E,0.808,173,1.876985e-06,3,Uncertain significance
1,2-113118022-G-A,113118022,p.Ala2Thr,p.Ala2Thr,2,Ala,E,0.808,173,2.502647e-06,4,Uncertain significance
2,2-113118025-T-A,113118025,p.Leu3Ile,p.Leu3Ile,3,Leu,I,0.672,131,6.243156e-07,1,NaN
3,2-113118026-T-C,113118026,p.Leu3Ser,p.Leu3Ser,3,Leu,I,0.672,131,1.247141e-06,2,NaN
4,2-113118028-G-C,113118028,p.Ala4Pro,p.Ala4Pro,4,Ala,C,0.642,95,6.224146e-07,1,NaN


In [2]:
# --- Colab cell 2: build a join key (protein_variant like A123T) from the RSA file ---

AA3_TO_AA1 = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G",
    "His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S",
    "Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Sec":"U","Pyl":"O"
}

def hgvs_p_to_protein_variant(hgvs_p: str):
    """
    Convert 'p.Ala2Ser' or 'Ala2Ser' -> 'A2S'
    Returns None if it can't parse a simple missense substitution.
    """
    if pd.isna(hgvs_p):
        return None
    s = str(hgvs_p).strip()
    s = s.replace("p.", "")  # handle 'p.Ala2Ser'

    m = re.fullmatch(r"([A-Za-z]{3})(\d+)([A-Za-z]{3})", s)
    if not m:
        return None

    wt3, pos, mut3 = m.group(1), m.group(2), m.group(3)
    wt1 = AA3_TO_AA1.get(wt3)
    mut1 = AA3_TO_AA1.get(mut3)
    if wt1 is None or mut1 is None:
        return None

    return f"{wt1}{pos}{mut1}"

# Prefer 'Protein Consequence' if present; else fall back to 'HGVS Consequence'
source_col = "Protein Consequence" if "Protein Consequence" in rsa.columns else "HGVS Consequence"

rsa["protein_variant"] = rsa[source_col].apply(hgvs_p_to_protein_variant)

# Keep only rows that parsed cleanly
rsa_parsed = rsa.dropna(subset=["protein_variant"]).copy()

print("Parsed variants:", rsa_parsed["protein_variant"].nunique(), "of", len(rsa), "rows")
rsa_parsed[["protein_variant", source_col]].head(10)


Parsed variants: 263 of 266 rows


,protein_variant,Protein Consequence
0,A2S,p.Ala2Ser
1,A2T,p.Ala2Thr
2,L3I,p.Leu3Ile
3,L3S,p.Leu3Ser
4,A4P,p.Ala4Pro
5,D5E,p.Asp5Glu
6,L6V,p.Leu6Val
7,L6F,p.Leu6Phe
8,Y7C,p.Tyr7Cys
9,E9K,p.Glu9Lys


In [5]:
def hgvs_p_to_protein_variant(hgvs_p: str):
    if pd.isna(hgvs_p):
        return None
    s = str(hgvs_p).strip().replace("p.", "")

    m = re.fullmatch(r"([A-Za-z]{3})(\d+)([A-Za-z]{3})", s)
    if not m:
        return None

    wt3, pos, mut3 = m.group(1).title(), m.group(2), m.group(3).title()
    wt1 = AA3_TO_AA1.get(wt3)
    mut1 = AA3_TO_AA1.get(mut3)
    if wt1 is None or mut1 is None:
        return None

    return f"{wt1}{pos}{mut1}"

rsa["protein_variant"] = rsa[source_col].apply(hgvs_p_to_protein_variant)
rsa_parsed = rsa.dropna(subset=["protein_variant"]).copy()

print("Parsed variants:", rsa_parsed["protein_variant"].nunique(), "of", len(rsa), "rows")


Parsed variants: 263 of 266 rows


In [7]:
def hgvs_p_to_protein_variant(hgvs_p: str):
    if pd.isna(hgvs_p):
        return None
    s = str(hgvs_p).strip().replace("p.", "")

    # Case A: 3-letter (Ala2Ser)
    m = re.fullmatch(r"([A-Za-z]{3})(\d+)([A-Za-z]{3})", s)
    if m:
        wt3, pos, mut3 = m.group(1).title(), m.group(2), m.group(3).title()
        wt1 = AA3_TO_AA1.get(wt3)
        mut1 = AA3_TO_AA1.get(mut3)
        return f"{wt1}{pos}{mut1}" if wt1 and mut1 else None

    # Case B: 1-letter (A2S)
    m = re.fullmatch(r"([ACDEFGHIKLMNPQRSTVWY])(\d+)([ACDEFGHIKLMNPQRSTVWY])", s)
    if m:
        return f"{m.group(1)}{m.group(2)}{m.group(3)}"

    return None

rsa["protein_variant"] = rsa[source_col].apply(hgvs_p_to_protein_variant)
rsa_parsed = rsa.dropna(subset=["protein_variant"]).copy()

print("Parsed variants:", rsa_parsed["protein_variant"].nunique(), "of", len(rsa), "rows")


Parsed variants: 263 of 266 rows


In [8]:
merged_all = rsa.merge(
    am[["protein_variant", "am_pathogenicity", "am_class"]],
    on="protein_variant",
    how="left"
)

print("Total RSA rows kept:", len(merged_all))
print("AM matched:", merged_all["am_pathogenicity"].notna().sum())

merged_all.head()


Total RSA rows kept: 266
AM matched: 229


,gnomAD ID,Position,HGVS Consequence,Protein Consequence,ResNum,WT_AA,AA,RSA,ACC,Allele Frequency,Allele Count,ClinVar Germline Classification,protein_variant,am_pathogenicity,am_class
0,2-113118022-G-T,113118022,p.Ala2Ser,p.Ala2Ser,2,Ala,E,0.808,173,1.876985e-06,3,Uncertain significance,A2S,NaN,NaN
1,2-113118022-G-A,113118022,p.Ala2Thr,p.Ala2Thr,2,Ala,E,0.808,173,2.502647e-06,4,Uncertain significance,A2T,NaN,NaN
2,2-113118025-T-A,113118025,p.Leu3Ile,p.Leu3Ile,3,Leu,I,0.672,131,6.243156e-07,1,NaN,L3I,NaN,NaN
3,2-113118026-T-C,113118026,p.Leu3Ser,p.Leu3Ser,3,Leu,I,0.672,131,1.247141e-06,2,NaN,L3S,NaN,NaN
4,2-113118028-G-C,113118028,p.Ala4Pro,p.Ala4Pro,4,Ala,C,0.642,95,6.224146e-07,1,NaN,A4P,NaN,NaN


In [9]:
unmatched = merged_all.loc[merged_all["am_pathogenicity"].isna(), "protein_variant"].dropna().unique()
print("Unmatched variants:", len(unmatched))
unmatched[:50]


Unmatched variants: 37


array(['A2S', 'A2T', 'L3I', 'L3S', 'A4P', 'D5E', 'L6V', 'L6F', 'Y7C',
       'E9K', 'E9V', 'G10R', 'G10E', 'G11S', 'G11D', 'G12R', 'G13R',
       'G13E', 'G13A', 'G15E', 'E16G', 'E16D', 'G17D', 'G17V', 'E18K',
       'D19G', 'N20H', 'N20S', 'A21T', 'A21D', 'A21V', 'A21G', 'D22G',
       'S23L', 'K24E', 'E25K', 'E25Q'], dtype=object)

In [10]:
# extract positions from A123T
merged_all["pos"] = pd.to_numeric(
    merged_all["protein_variant"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

am_pos = pd.to_numeric(am["protein_variant"].str.extract(r"(\d+)")[0], errors="coerce")

print("RSA position min/max:", merged_all["pos"].min(), merged_all["pos"].max())
print("AM position  min/max:", am_pos.min(), am_pos.max())

# show unmatched positions distribution
unmatched_pos = merged_all.loc[merged_all["am_pathogenicity"].isna(), "pos"]
unmatched_pos.describe()


RSA position min/max: 2 177
AM position  min/max: 1 177


,pos
count,37.000000
mean,13.783784
std,7.102954
min,2.000000
25%,9.000000
50%,13.000000
75%,20.000000
max,25.000000


In [11]:
# Build a set of all AM variants for fast membership checks
am_set = set(am["protein_variant"])

# For each unmatched variant, see if same position+mut exists but with different WT AA
def alt_wt_hits(var):
    m = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(var))
    if not m:
        return None
    wt, pos, mut = m.group(1), m.group(2), m.group(3)
    alts = [f"{a}{pos}{mut}" for a in "ACDEFGHIKLMNPQRSTVWY" if a != wt]
    hits = [v for v in alts if v in am_set]
    return hits[:5] if hits else None

u = pd.DataFrame({"protein_variant": unmatched})
u["possible_alt_wt_matches"] = u["protein_variant"].apply(alt_wt_hits)
u[u["possible_alt_wt_matches"].notna()].head(20)


,protein_variant,possible_alt_wt_matches
0,A2S,[E2S]
1,A2T,[E2T]
3,L3S,[I3S]
4,A4P,[C4P]
5,D5E,[R5E]
6,L6V,[G6V]
7,L6F,[G6F]
8,Y7C,[L7C]
9,E9K,[S9K]
10,E9V,[S9V]
